# Avatar Falante PT-BR - Colab (GPU T4)

1. `Ambiente > Alterar tipo de ambiente de execucao > GPU T4`
2. Rode as celulas em ordem. A ultima gera o link publico do Gradio (`share=True`).

> Instalacao do SadTalker no Colab depende das versoes de Python/PyTorch do momento; se alguma celula falhar, veja a secao *Problemas conhecidos* do README.

In [ ]:
!nvidia-smi
import torch; print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Ative a GPU T4 no menu de ambiente de execucao.'

In [ ]:
# Envie o arquivo meu-avatar.zip (zip da pasta do projeto, sem .venv)
from google.colab import files
import zipfile, os
up = files.upload()
nome = next(iter(up))
zipfile.ZipFile(nome).extractall('/content/meu-avatar')
%cd /content/meu-avatar
!ls

In [ ]:
!apt-get -qq install -y ffmpeg
!pip -q install 'gradio>=6.0' 'pillow>=12.3.0,<13.0' edge-tts 'rembg[gpu]' onnxruntime-gpu imageio-ffmpeg diffusers transformers accelerate huggingface_hub

In [ ]:
# SadTalker + GFPGAN (checkpoints ~ alguns GB)
!git clone --depth 1 https://github.com/OpenTalker/SadTalker.git
%cd SadTalker
!pip -q install face_alignment imageio imageio-ffmpeg librosa resampy pydub kornia yacs scikit-image basicsr facexlib gfpgan safetensors av 'numpy<2' 'pillow>=12.3.0,<13.0'
!bash scripts/download_models.sh
%cd ..

In [ ]:
# Correcoes comuns de compatibilidade do SadTalker com numpy/torchvision recentes
import re, pathlib
for f in pathlib.Path('SadTalker').rglob('*.py'):
    t = f.read_text()
    n = t.replace('np.float,', 'float,').replace('np.float)', 'float)').replace('np.int)', 'int)')
    if n != t: f.write_text(n)
# basicsr importa torchvision.transforms.functional_tensor (removido nas versoes novas)
import site, glob
for f in glob.glob(site.getsitepackages()[0] + '/basicsr/data/degradations.py'):
    t = open(f).read().replace('torchvision.transforms.functional_tensor', 'torchvision.transforms.functional')
    open(f, 'w').write(t)

### Teste sem interface (diagnostico)
Rode estas duas celulas **antes** de subir o app. Elas mostram o erro completo do SadTalker, sem depender do Gradio. Se `Checkpoints` ou `SadTalker` aparecerem como ausentes, rode de novo as celulas de instalacao.

In [ ]:
# Diagnostico: GPU, SadTalker, checkpoints e versoes
%cd /content/meu-avatar
import sys, importlib
for m in [m for m in sys.modules if m in ("animador","voz","config","compositor","gerador_cenario","app")]:
    del sys.modules[m]          # garante o codigo do zip novo
import os
os.environ["SADTALKER_DIR"] = "/content/meu-avatar/SadTalker"
import animador
print(animador.diagnostico())
import torch, numpy, PIL, gradio
print("torch", torch.__version__, "| numpy", numpy.__version__, "| pillow", PIL.__version__, "| gradio", gradio.__version__)

In [ ]:
# Teste do SadTalker direto, com log completo (sem Gradio)
import glob, subprocess, sys, pathlib
import voz
raiz = animador.localizar_sadtalker()
assert raiz, "SadTalker nao encontrado: rode a celula de instalacao."
assert animador.checkpoints_ok(raiz), f"Checkpoints ausentes em {raiz/'checkpoints'}: rode scripts/download_models.sh."

# Imagem: a ultima enviada pelo app, senao um exemplo do proprio SadTalker
candidatas = glob.glob("avatares/avatar_atual.png") + sorted(glob.glob(str(raiz / "examples/source_image/*.png")))
assert candidatas, "Nenhuma imagem de teste encontrada."
imagem = pathlib.Path(candidatas[0]).resolve()
print("Imagem de teste:", imagem)

audio = voz.gerar_audio("Ola! Este e um teste de animacao do avatar. [pausa] Funcionou?", "Francisca (feminina)")
saida = pathlib.Path("temp/teste_sadtalker").resolve()
cmd = [sys.executable, "inference.py", "--driven_audio", str(audio.resolve()),
       "--source_image", str(imagem), "--result_dir", str(saida),
       *animador.flags_naturalidade(movimento_cabeca=True, restaurar_face=True)]
print("Comando:", " ".join(cmd), "\n")
p = subprocess.run(cmd, cwd=raiz, capture_output=True, text=True)
print("codigo de saida:", p.returncode)
print("--- STDOUT (final) ---"); print("\n".join(p.stdout.splitlines()[-25:]))
print("--- STDERR (final) ---"); print("\n".join(p.stderr.splitlines()[-40:]))

videos = sorted(saida.rglob("*.mp4"))
if p.returncode == 0 and videos:
    from IPython.display import Video, display
    print("OK:", videos[-1]); display(Video(str(videos[-1]), embed=True, width=400))
else:
    print("\nFALHOU. Copie o bloco STDERR acima e envie para correcao.")

In [ ]:
%cd /content/meu-avatar
# Sobe o app com link publico. O SadTalker e detectado automaticamente.
import os, sys
for m in [m for m in sys.modules if m in ('animador','voz','config','compositor','gerador_cenario','app')]:
    del sys.modules[m]
os.environ['SADTALKER_DIR'] = '/content/meu-avatar/SadTalker'
from app import construir
demo = construir()
demo.queue().launch(share=True, debug=True)

### Usar o Colab como motor de animacao do app local
Copie a URL publica (`https://xxxx.gradio.live`) e, na maquina local:
```bash
REMOTE_ANIMADOR_URL=https://xxxx.gradio.live .venv/bin/python app.py
```